# NB05 공개키 서명 — 열쇠 둘로 권한을 나누기

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB05-1: 장난감 타원곡선 y^2 = x^3 + 7 (mod 97) — 점 덧셈, 두 배, 스칼라 곱 (공부용, 실사용 금지)
p, a, b = 97, 0, 7          # secp256k1과 같은 모양(a=0, b=7), 크기만 작은 곡선
O = None                     # 무한원점(항등원)

def on_curve(P):
    if P is O: return True
    x, y = P
    return (y * y - (x ** 3 + a * x + b)) % p == 0

def inv(z): return pow(z, p - 2, p)              # 곱셈 역원 (페르마 소정리)

def add(P, Q):
    if P is O: return Q
    if Q is O: return P
    (x1, y1), (x2, y2) = P, Q
    if x1 == x2 and (y1 + y2) % p == 0: return O   # P + (-P) = O
    if P == Q: lam = (3 * x1 * x1 + a) * inv(2 * y1) % p
    else:      lam = (y2 - y1) * inv(x2 - x1) % p
    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)

def mul(k, P):                                    # 두 배하고-더하기 (double-and-add)
    R = O
    while k:
        if k & 1: R = add(R, P)
        P = add(P, P)
        k >>= 1
    return R

points = [(x, y) for x in range(p) for y in range(p) if on_curve((x, y))]
print("곡선 위의 점 개수(무한원점 포함) n =", len(points) + 1)
G = min(points)                                   # x가 가장 작은 점을 기준점으로
print("기준점 G =", G)
print("2G =", add(G, G), " 3G =", mul(3, G), " 79G =", mul(79, G), "(= 무한원점이면 None)")
d = 23                                            # 장난감 개인키 — 실사용 금지
Q = mul(d, G)
print("개인키 d = %d → 공개키 Q = [d]G = %s" % (d, Q))
found = next(k for k in range(1, 79) if mul(k, G) == Q)
print("전수 탐색으로 Q에서 d를 되찾음: d =", found, "(79개뿐이라 즉시 — 실제 곡선은 2^256 규모)")

곡선 위의 점 개수(무한원점 포함) n = 79
기준점 G = (1, 28)
2G = (68, 81)  3G = (53, 38)  79G = None (= 무한원점이면 None)
개인키 d = 23 → 공개키 Q = [d]G = (61, 28)
전수 탐색으로 Q에서 d를 되찾음: d = 23 (79개뿐이라 즉시 — 실제 곡선은 2^256 규모)


In [ ]:
# NB05-2: 장난감 ECDSA — 서명 생성·검증, 한 비트 바꾸면 실패, (r, n-s)도 유효 (공부용, 실사용 금지)
import hashlib
n = 79                                            # 기준점 G의 위수

def inv_n(z): return pow(z, n - 2, n)

def msg_int(M: bytes) -> int:
    """메시지 해시를 정수로. n이 작으니 SHA-256 결과를 n으로 줄인다(설명용 단순화)."""
    return int.from_bytes(hashlib.sha256(M).digest(), "big") % n

def sign(d, M, k):
    e = msg_int(M)
    R = mul(k, G)
    r = R[0] % n
    s = inv_n(k) * (e + r * d) % n
    assert r != 0 and s != 0
    return (r, s)

def verify(Q, M, sig):
    r, s = sig
    if not (1 <= r <= n - 1 and 1 <= s <= n - 1): return False
    e = msg_int(M)
    u = e * inv_n(s) % n
    v = r * inv_n(s) % n
    R1 = add(mul(u, G), mul(v, Q))
    return R1 is not O and R1[0] % n == r

M = b"Pay 1 coin to Bob"
k = 17                                            # 일회용 비밀수 — 설명을 위해 고정값(실제로는 안전한 난수)
sig = sign(d, M, k)
print("메시지:", M, "| 서명 (r, s) =", sig)
print("정상 검증            :", verify(Q, M, sig))
print("메시지 한 글자 변경    :", verify(Q, b"Pay 2 coin to Bob", sig))
print("다른 사람의 공개키로   :", verify(mul(24, G), M, sig))
r, s = sig
print("(r, n-s) =", (r, (n - s) % n), "도 유효한가:", verify(Q, M, (r, (n - s) % n)), " ← ECDSA 가변성")

메시지: b'Pay 1 coin to Bob' | 서명 (r, s) = (17, 45)
정상 검증            : True
메시지 한 글자 변경    : False
다른 사람의 공개키로   : False
(r, n-s) = (17, 34) 도 유효한가: True  ← ECDSA 가변성


In [ ]:
# NB05-3: 비밀수 k를 두 번 쓰면 — 공개된 서명 둘로 개인키를 계산한다 (공부용)
M1, M2 = b"Pay 1 coin to Bob", b"Pay 5 coins to Carol"
k_reused = 31
(r1, s1), (r2, s2) = sign(d, M1, k_reused), sign(d, M2, k_reused)
e1, e2 = msg_int(M1), msg_int(M2)
print("두 서명의 r이 같다:", r1 == r2, "(r =", r1, ")")
k_rec = (e1 - e2) * inv_n(s1 - s2) % n
d_rec = (s1 * k_rec - e1) * inv_n(r1) % n
print("되찾은 k =", k_rec, "| 되찾은 개인키 d =", d_rec, "| 실제 d =", d)

두 서명의 r이 같다: True (r = 52 )
되찾은 k = 31 | 되찾은 개인키 d = 23 | 실제 d = 23


In [ ]:
# NB05-4: 실제 secp256k1 매개변수(SEC 2 §2.4.1) — 표준 라이브러리만으로 확인 (공부용, 실사용 금지)
import hashlib
p = 2**256 - 2**32 - 2**9 - 2**8 - 2**7 - 2**6 - 2**4 - 1
a, b = 0, 7
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
G = (Gx, Gy)
print("p 는 16진수로", hex(p).upper()[2:])
print("G 가 곡선 위에 있는가:", on_curve(G))
print("[n]G 가 무한원점인가 :", mul(n, G) is O)
# 압축 표현 02||x 에서 y 복원: p ≡ 3 (mod 4) 이므로 y = (x^3+7)^((p+1)/4). 접두 02 는 y 가 짝수라는 뜻.
y_cand = pow((Gx**3 + 7) % p, (p + 1) // 4, p)
y_even = y_cand if y_cand % 2 == 0 else p - y_cand
print("압축 표현에서 복원한 y == 비압축 표현의 y :", y_even == Gy)

d_toy = 1                                         # 공부용 개인키 1 → 공개키는 G 자신. 실제 키 생성이 아니다.
Qx, Qy = mul(d_toy, G)
compressed = bytes([2 + (Qy & 1)]) + Qx.to_bytes(32, "big")
print("d=1 의 압축 공개키:", compressed.hex())
h160 = hashlib.new("ripemd160", hashlib.sha256(compressed).digest()).digest()
print("RIPEMD160(SHA256(K)) 20바이트:", h160.hex())
print("비압축 65바이트 공개키의 첫 바이트는 0x04 — 12장 제네시스 출력의 공개키가 이 꼴이다.")

p 는 16진수로 FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
G 가 곡선 위에 있는가: True
[n]G 가 무한원점인가 : True
압축 표현에서 복원한 y == 비압축 표현의 y : True
d=1 의 압축 공개키: 0279be667ef9dcbbac55a06295ce870b07029bfcdb2dce28d959f2815b16f81798
RIPEMD160(SHA256(K)) 20바이트: 751e76e8199196d454941c45d1b3a323f1433bd6
비압축 65바이트 공개키의 첫 바이트는 0x04 — 12장 제네시스 출력의 공개키가 이 꼴이다.
